# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 Kilifi County Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL at:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

For each record set, we will print its `@id`, name, and fields (with their `@id`). All access is via `@id` references as required.

In [ ]:
# List available record sets and fields, referencing by @id
record_sets = dataset.record_sets
print("Record Sets in the dataset (referenced by @id):")
for rs in record_sets:
    print(f"- Record Set @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    fields = rs.get('field', [])
    if fields:
        print("  Fields:")
        for field in fields:
            if isinstance(field, dict):
                print(f"    - Field @id: {field['@id']} | Name: {field.get('name', 'N/A')}")
            else:
                print(f"    - Field @id: {field}")
    else:
        print("  No fields listed.")
    print("\n")
# Save the list of record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

If there is only one record set, it will be used for all extraction below. Otherwise, you may select a record set by its `@id`.

In [ ]:
# Extract data from each record set by @id
dataframes = {}

# Loop through all record set @ids
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for Record Set @id: {record_set_id}")
        print("Columns (fields by @id):", df.columns.tolist())
        print(df.head())
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")
# Choose one record set @id for EDA
main_record_set_id = record_set_ids[0]
print("\nUsing main record set @id:", main_record_set_id)
df_main = dataframes[main_record_set_id]

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
All fields are referenced by their `@id`, as required.

In [ ]:
# Identify candidate numeric and group fields by their @id
import numpy as np

numeric_field_id = None
group_field_id = None
# Try to infer: try to find a numeric field and a categorical/group field
for col in df_main.columns:
    if np.issubdtype(df_main[col].dtype, np.number):
        numeric_field_id = col
    if group_field_id is None and (df_main[col].dtype == 'object'):
        group_field_id = col
    if numeric_field_id and group_field_id:
        break
print(f"Chosen numeric field @id: {numeric_field_id}")
print(f"Chosen group field @id: {group_field_id}")

# Set a filtering threshold
if numeric_field_id:
    threshold = df_main[numeric_field_id].mean()
    filtered_df = df_main[df_main[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # If group field exists, show grouped aggregation
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using `matplotlib` and `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot for numeric field
if numeric_field_id:
    plt.figure(figsize=(8,5))
    sns.histplot(df_main[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

# Boxplot for numeric field grouped by group field
if numeric_field_id and group_field_id:
    plt.figure(figsize=(10,6))
    sns.boxplot(x=df_main[group_field_id], y=df_main[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset schema and content were loaded via the Croissant URL with `mlcroissant`, referencing all entities by their `@id`.
- A main record set was examined; fields were referenced via their `@id`, and data was filtered, normalized, and grouped.
- Visualization illustrated the distribution and grouping of the numeric field.
- This exploratory workflow can be extended to additional record sets and analytic goals using the standardized schema provided.